In [ ]:
import Pkg
Pkg.activate("..")
Pkg.instantiate()

using CSV
using DataFrames
using Plots

In [ ]:
dir = "../data/results/exp4_memory"
df = CSV.read(joinpath(dir, "exp4_memory.csv"), DataFrame)
base = filter(r -> r.Impl == "base", df)
opt  = filter(r -> r.Impl == "opt", df)
names_ = base.Dataset
x = 1:length(names_)

# Peak RAM của tiến trình và phần RAM tăng thêm do thuật toán (Δ = peak - baseline)
for (col, ttl, fname) in [(:Peak_RSS_MB, "Peak RAM (whole process)", "plot_peak_rss.png"),
                          (:Delta_RSS_MB, "Extra peak RAM used by mining (Δ RSS)", "plot_delta_rss.png"),
                          (:Alloc_MB, "Total allocated memory (GC pressure)", "plot_alloc.png")]
    p = bar(x .- 0.2, base[!, col], bar_width = 0.4, label = "Basic implementation")
    bar!(p, x .+ 0.2, opt[!, col], bar_width = 0.4, label = "Optimized implementation")
    plot!(p, xticks = (x, names_), title = ttl, ylabel = "MB", legend = :topleft, grid = true)
    display(p)
    savefig(p, joinpath(dir, fname))
end

In [ ]:
# Kích thước FP-tree ban đầu: tổng MB và số byte trung bình mỗi node
bytes_per_node = base.Tree_MB .* 1024^2 ./ max.(base.Tree_Nodes, 1)
p = bar(x, bytes_per_node, label = "Basic FP-tree", bar_width = 0.6,
        xticks = (x, names_), title = "Memory per FP-tree node", ylabel = "Bytes / node", grid = true)
display(p)
savefig(p, joinpath(dir, "plot_bytes_per_node.png"))

select(base, :Dataset, :Minsup, :Tree_Nodes, :Tree_MB, :DB_MB, :Peak_RSS_MB, :Delta_RSS_MB)

In [ ]:
# SPMF (nếu có): JVM heap do SPMF tự đo - chỉ để tham khảo, không cùng thước đo với RSS của Julia
spmf = filter(r -> r.Impl == "spmf", df)
spmf